# Agent 测试 Notebook

这个 notebook 用来逐个测试 `agent/agents` 里的每个 Agent，并在最后跑完整的 Orchestrator 链路。建议从上到下执行。

In [12]:
import importlib
import json
import sys
from dataclasses import asdict, is_dataclass
from pathlib import Path
from typing import Any

ROOT = Path.cwd()
if ROOT.name == "agent":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import agent.tools.data_loader as data_loader_module
import agent.tools as tools_module
import agent.agents.context_agent as context_agent_module
import agent.agents.forecast_agent as forecast_agent_module
import agent.agents.generation_agent as generation_agent_module
import agent.agents.intent_parser as intent_parser_module
import agent.agents.search_agent as search_agent_module

importlib.reload(data_loader_module)
importlib.reload(tools_module)
importlib.reload(context_agent_module)
importlib.reload(forecast_agent_module)
importlib.reload(generation_agent_module)
importlib.reload(intent_parser_module)
importlib.reload(search_agent_module)

from agent.models import AgentRequest, UserType
from agent.agents.context_agent import ContextAgent
from agent.agents.forecast_agent import ForecastAgent
from agent.agents.generation_agent import GenerationAgent
from agent.agents.intent_parser import IntentParser
from agent.agents.search_agent import SearchAgent
from agent.orchestrator import Orchestrator

TEST_QUERY = "我想在 2026-07-25 去 Salzburg，什么时候出发最好？"
TEST_DATE = "2026-07-25"
TEST_ROAD = "A8"
TEST_DESTINATION = "Salzburg"
TEST_HOURS = [7, 8, 9, 16, 17, 18]


def to_jsonable(value: Any) -> Any:
    if is_dataclass(value):
        return to_jsonable(asdict(value))
    if isinstance(value, dict):
        return {key: to_jsonable(item) for key, item in value.items()}
    if isinstance(value, list):
        return [to_jsonable(item) for item in value]
    if hasattr(value, "value"):
        return value.value
    return value


def show(title: str, value: Any) -> None:
    print(f"\n{'=' * 10} {title} {'=' * 10}")
    if isinstance(value, str):
        print(value)
    else:
        print(json.dumps(to_jsonable(value), ensure_ascii=False, indent=2))


def show_response(name: str, response) -> None:
    show(name, {
        "success": response.success,
        "message": response.message,
        "data": response.data,
    })

print("Agent test environment ready")

Agent test environment ready


## 1. IntentParser 测试

使用关键词降级模式测试意图解析，不依赖 LLM API key。

In [2]:
intent_parser = IntentParser(use_llm=False)
parsed_intent = intent_parser.parse(TEST_QUERY, UserType.TRAVELER)

show("IntentParser", {
    "persona_type": parsed_intent.persona_type,
    "user_type": parsed_intent.user_type,
    "core_question": parsed_intent.core_question,
    "destination": parsed_intent.destination,
    "road": parsed_intent.road,
    "intent": parsed_intent.intent,
    "time_range": parsed_intent.time_range,
    "data_requirements": parsed_intent.data_requirements,
    "trip_plan": parsed_intent.trip_plan,
})


========== IntentParser ==========
{
  "persona_type": "tourist",
  "user_type": "traveler",
  "core_question": "Tell me what I should do.",
  "destination": "salzburg",
  "road": "A8",
  "intent": "plan",
  "time_range": {
    "type": "custom",
    "start_date": "2026-07-25",
    "end_date": "2026-07-25",
    "duration_days": 1,
    "description": "2026-07-25"
  },
  "data_requirements": {
    "time_range": {
      "type": "custom",
      "start_date": "2026-07-25",
      "end_date": "2026-07-25",
      "duration_days": 1,
      "description": "2026-07-25"
    },
    "granularity": "hourly",
    "hours": [
      7,
      8,
      9,
      10,
      11,
      12,
      13,
      14,
      15,
      16,
      17,
      18,
      19
    ]
  },
  "trip_plan": {
    "trip_type": "round_trip",
    "stay_days": 1
  }
}


## 2. ForecastAgent 测试

分别测试小时级预测和日级预测。小时级用于单日出发建议，日级用于日历/长时间范围。

In [3]:
forecast_agent = ForecastAgent()

hourly_request = AgentRequest(
    query="forecast hourly test",
    date=TEST_DATE,
    road=TEST_ROAD,
    destination=TEST_DESTINATION,
    hours=TEST_HOURS,
    granularity="hourly",
)
hourly_forecast_result = await forecast_agent.process(hourly_request)
show_response("ForecastAgent hourly", hourly_forecast_result)


========== ForecastAgent hourly ==========
{
  "success": true,
  "message": "",
  "data": {
    "mode": "hourly",
    "forecast": {
      "date": "2026-07-25",
      "road": "A8",
      "site_id": "A8_default",
      "predictions": [
        {
          "hour": 7,
          "kfz_h_p10": 2088.0190519585394,
          "kfz_h_p50": 2450.665253467299,
          "kfz_h_p90": 2818.6614530746465,
          "sv_h": 287.6409731211398,
          "v_kfz": 127.5353384621389,
          "congestion_score": 23.4,
          "congestion_level": "light"
        },
        {
          "hour": 7,
          "kfz_h_p10": 1594.9503025184742,
          "kfz_h_p50": 2098.9142870473383,
          "kfz_h_p90": 2439.232773056983,
          "sv_h": 65.10422164135021,
          "v_kfz": 103.36669997920768,
          "congestion_score": 25.7,
          "congestion_level": "light"
        },
        {
          "hour": 7,
          "kfz_h_p10": 1598.9207656457709,
          "kfz_h_p50": 2077.848647105513,
         

In [4]:
daily_request = AgentRequest(
    query="forecast daily test",
    date=TEST_DATE,
    road=TEST_ROAD,
    destination=TEST_DESTINATION,
    start_date="2026-07-25",
    end_date="2026-07-31",
    granularity="daily",
    include_factors=True,
)
daily_forecast_result = await forecast_agent.process(daily_request)
show_response("ForecastAgent daily", daily_forecast_result)


========== ForecastAgent daily ==========
{
  "success": true,
  "message": "",
  "data": {
    "mode": "daily",
    "daily_forecasts": [
      {
        "date": "2026-07-25",
        "road": "A8",
        "direction": "Mch",
        "site_id": "A8_Mch_MQB25_Mch_H",
        "site_name": "MQB25_Mch_H",
        "kfz_h_p10": 58081.0,
        "kfz_h_p50": 70603.0,
        "kfz_h_p90": 83239.0,
        "sv_h_pred": 4287.0,
        "v_kfz_pred": 120.8,
        "interval_width": 25158.0,
        "relative_interval_width": 0.428096,
        "congestion_score": 23.3,
        "congestion_level": "light",
        "reasons": [
          {
            "name": "Historical Traffic Baseline",
            "value": 67.5
          },
          {
            "name": "Weather and Temperature",
            "value": 12.6
          },
          {
            "name": "Date and Time Pattern",
            "value": 9.1
          },
          {
            "name": "Special Events",
            "value": 4.8
      

## 3. ContextAgent 测试

测试离线上下文数据，包括天气、假期、活动、施工、气温/路温和历史小时交通。

In [13]:
context_agent = ContextAgent()
context_request = AgentRequest(
    query="context test",
    date=TEST_DATE,
    road=TEST_ROAD,
    destination=TEST_DESTINATION,
    start_date="2026-07-25",
    end_date="2026-07-27",
    hours=[7, 8, 9],
)
context_result = await context_agent.process(context_request)
show("ContextAgent status", {
    "success": context_result.success,
    "message": context_result.message,
})

if context_result.success:
    context_payload = context_result.data.get("context", {})
    context_counts = {
        key: len(value) if isinstance(value, list) else None
        for key, value in context_payload.items()
        if key != "historical_same_period"
    }
    historical_payload = context_payload.get("historical_same_period", {})
    historical_counts = {
        key: len(value) if isinstance(value, list) else None
        for key, value in historical_payload.items()
    }
    history_factor_types = [
        factor.type for factor in context_result.data.get("factors", [])
        if factor.source == "context_history"
    ]
    show("Context summary", context_result.data.get("summary"))
    show("Context row counts", context_counts)
    show("Historical same-period row counts", historical_counts)
    show("Historical same-period factor types", history_factor_types)


========== ContextAgent status ==========
{
  "success": true,
  "message": ""
}

========== Context summary ==========
{
  "weather_days": 3,
  "holiday_days": 3,
  "construction_days": 3,
  "event_days": 3,
  "temperature_hours": 9,
  "traffic_records": 162,
  "avg_speed_kmh": 107.7,
  "max_hourly_volume": 4992.0,
  "min_air_temp_c": 14.8,
  "max_air_temp_c": 18.3,
  "min_road_temp_c": 18.2,
  "max_road_temp_c": 26.5,
  "historical_same_period": {
    "years": [
      "2023",
      "2024",
      "2025"
    ],
    "weather_days": 9,
    "holiday_days": 9,
    "construction_days": 0,
    "event_days": 9,
    "temperature_hours": 27,
    "traffic_records": 162,
    "avg_speed_kmh": 107.7,
    "max_hourly_volume": 4992.0
  }
}

========== Context row counts ==========
{
  "weather": 3,
  "holiday": 3,
  "events": 3,
  "construction": 3,
  "temperature_road": 9,
  "hourly_traffic": 162
}

========== Historical same-period row counts ==========
{
  "weather": 9,
  "holiday": 9,
  "events"

## 4. SearchAgent 测试

测试 Tavily 搜索 Agent。它会按天气、施工、活动、事故四类生成搜索任务；如果没有 Tavily key，会返回 search plan 和错误提示。

In [6]:
search_agent = SearchAgent()
search_request = AgentRequest(
    query="search real-time factors test",
    date=TEST_DATE,
    road=TEST_ROAD,
    destination=TEST_DESTINATION,
)
search_result = await search_agent.process(search_request)
show_response("SearchAgent", search_result)

if search_result.success:
    show("Search plan", search_result.data.get("search_plan"))
    show("Search errors", search_result.data.get("errors"))


========== SearchAgent ==========
{
  "success": true,
  "message": "",
  "data": {
    "factors": [
      {
        "type": "weather",
        "name": "天气预报搜索",
        "description": "On 2026-07-25, expect heavy snow, freezing rain, and black ice in Munich, Rosenheim, and Salzburg. Severe weather warnings may be in effect. Drive cautiously due to potential road disruptions.",
        "impact": "high",
        "source": "search"
      },
      {
        "type": "construction",
        "name": "A8 施工/封路搜索",
        "description": "The A8 Autobahn between Munich and Salzburg will have roadworks closures in July 2026. The project aims to upgrade the highway with a new alignment. Construction is expected to cause traffic disruptions.",
        "impact": "high",
        "source": "search"
      },
      {
        "type": "event",
        "name": "沿线活动搜索",
        "description": "On July 25, 2026, the Auer Dult festival in Munich will take place. Major events include the FREE & EASY Festiv

## 5. GenerationAgent 测试

使用前面 Agent 的输出生成最终建议。这里复用 `IntentParser` 的解析结果、`ForecastAgent` 的预测和 Context/Search factors。

In [7]:
generation_agent = GenerationAgent()

generation_forecast = None
if hourly_forecast_result.success:
    generation_forecast = hourly_forecast_result.data.get("forecast")

context_factors = context_result.data.get("factors", []) if context_result.success else []
search_factors = search_result.data.get("factors", []) if search_result.success else []

generation_result = await generation_agent.process(
    request=hourly_request,
    parsed_intent=parsed_intent,
    forecast=generation_forecast,
    context_factors=context_factors,
    search_factors=search_factors,
)
show_response("GenerationAgent", generation_result)

if generation_result.success:
    show("Generated advice", generation_result.data.get("advice"))


========== GenerationAgent ==========
{
  "success": true,
  "message": "",
  "data": {
    "persona": "tourist",
    "core_question": "Tell me what I should do.",
    "time_range_type": "short",
    "advice": "### 🧳 驾驶建议\n\n**去Salzburg？**\n\n#### 🚗 去程\n\n由于2026-07-25 A8 施工，拥堵风险较高。\n\n✅ **建议**: 07:30 前出发，可以避开大部分车流\n\n⏱️ 预计行程: 1小时44分钟\n\n#### 🔙 返程\n\n**返程时间建议**:\n- 周日返程：建议 12:00 前出发\n- 避开 15:00-19:00 返城高峰",
    "data": {
      "congestion_level": "high",
      "recommended_time": "07:30",
      "trip_type": "round_trip"
    },
    "factors": [
      {
        "type": "weather",
        "name": "2026-07-25 低能见度",
        "description": "低能见度小时数约 2 小时",
        "impact": "moderate",
        "source": "context"
      },
      {
        "type": "weather",
        "name": "2026-07-26 低能见度",
        "description": "低能见度小时数约 2.25 小时",
        "impact": "moderate",
        "source": "context"
      },
      {
        "type": "school_holiday",
        "name": "2026-07-25 学校假期",
        "descrip

## 6. 完整 Agent 链路测试

这里直接调用 `Orchestrator`，完整执行 IntentParser → ForecastAgent / ContextAgent / SearchAgent → GenerationAgent。

In [8]:
orchestrator = Orchestrator()
chain_result = await orchestrator.process(TEST_QUERY, UserType.TRAVELER)

show("Full chain success", chain_result.get("success"))
show("Full chain persona", chain_result.get("persona"))
show("Full chain time range", chain_result.get("time_range"))
show("Full chain advice", chain_result.get("advice"))
show("Full chain raw keys", list(chain_result.get("raw", {}).keys()))

[Orchestrator] Persona: tourist
[Orchestrator] Core Question: Tell me what I should do.
[Orchestrator] Time Range: custom (2026-07-25)
[Orchestrator]   - Start: 2026-07-25
[Orchestrator]   - End: 2026-07-25
[Orchestrator]   - Duration: 1 days
[Orchestrator] Granularity: hourly
[Orchestrator] Trip Type: round_trip
[Orchestrator] Stay Days: 1
[Orchestrator] Running agents: forecast, context, search

========== Full chain success ==========
true

========== Full chain persona ==========
{
  "type": "tourist",
  "core_question": "Tell me what I should do."
}

========== Full chain time range ==========
{
  "type": "custom",
  "start_date": "2026-07-25",
  "end_date": "2026-07-25",
  "duration_days": 1,
  "description": "2026-07-25",
  "granularity": "hourly"
}

========== Full chain advice ==========
### 🧳 驾驶建议

**去Salzburg？**

#### 🚗 去程

由于2026-07-25 A8 施工，拥堵风险较高。

✅ **建议**: 07:30 前出发，可以避开大部分车流

⏱️ 预计行程: 1小时44分钟

#### 🔙 返程

**返程时间建议**:
- 周日返程：建议 12:00 前出发
- 避开 15:00-19:00 返城高峰

==========